# Workstream 4: Issue [L7] — Run All Lenses and Write Results (Matteo)

Notebook for developing, validating, and executing the full evaluation of all three lenses:
1. **Standard Logit Lens** (nostalgebraist, 2020)
2. **Tuned Lens** (Belrose et al., 2023)
3. **Jacobian Lens** (Gurnee et al., 2026)

across the evaluation datasets:
- **`gpt2`** on **`pud_en`**
- **`gpt2`** on **`wikitext`** (test split, 256-token windows)
- **`geppetto`** on **`pud_it`**

---

### Contract Specifications (`docs/CONTRACTS.md` Sections 9.1, 9.3, 9.4 & Issue #25)
1. **Positions & Prediction Targets**:
   - Prediction at position $t$ is compared with the target token at $t + 1$.
   - Last position $T-1$ of each sequence has no target and is skipped.
   - On PUD (UD data), sentences are encoded with `encode_words(..., add_bos=True)`.
   - For the **UPOS Bridge**, position for word $w$ is the **last subword of $w$**, and gold label is `next_upos` of $w$.
2. **Memory Efficiency**:
   - Forward pass once per batch.
   - For each lens and layer, compute logits and immediately reduce to metrics (`top1_acc`, `entropy`, `kl`, `upos_acc`).
   - Free logits immediately before moving to the next layer.
3. **Outputs** (6 Parquet files):
   - 3 Aggregate files: `results/lens/{model}__{dataset}.parquet`
     - Columns: `model, dataset, lens, layer, L, tau, top1_acc, entropy_mean, kl_mean, upos_top1_acc, n_positions`
   - 3 Per-token files: `results/lens/tokens/{model}__{dataset}.parquet` (first 100 sequences)
     - Columns: `seq_id, pos, token, target_token, lens, layer, top1_token, top1_prob, entropy, kl`
4. **Sanity Criteria**:
   - At layer $L$, all three lenses give the exact same `top1_acc` (matching the base model).
   - At layer $L$, $\text{KL} = 0.0$.

## 1. Environment Setup and Imports

In [1]:
import gc
import sys
import time
from pathlib import Path

import pandas as pd
import torch

# Locate repository root and add src/ to sys.path
cwd = Path.cwd().resolve()
ROOT_DIR = cwd if (cwd / "src").exists() else cwd.parent
SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lib.data import load_model
from lib.paths import ROOT, lens_result_path
from lib.registry import get_model_spec
from lib.word_table import load_word_table
from ws4_lens.evaluate import evaluate_lenses, prepare_pud_batches, prepare_wikitext_batches
from ws4_lens.jacobian_lens import load_jacobian_checkpoint
from ws4_lens.tuned_lens import load_official_tuned_lens, load_tuned_lens_checkpoint
from ws4_lens.upos_bridge import build_upos_lexicon

print(f"Repository root: {ROOT_DIR}")
print(f"PyTorch version: {torch.__version__}")
print(f"MPS available:   {torch.backends.mps.is_available()}")

/Users/matteosalvini/Desktop/Magistrale/HLT/HLT_Project-LLMs_on_Probing_and_Logit_Lens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository root: /Users/matteosalvini/Desktop/Magistrale/HLT/HLT_Project-LLMs_on_Probing_and_Logit_Lens
PyTorch version: 2.15.0
MPS available:   True


## Run 1 (`gpt2`, `pud_en`) — ✅ ALREADY COMPLETED

This run was executed via `scripts/32_run_lenses.py` and completed in ~402s.

**Sanity check at layer L=12:**
```
     lens  top1_acc  kl_mean  upos_top1_acc
    logit  0.250635      0.0        0.39782
    tuned  0.250635      0.0        0.39782
 jacobian  0.250635      0.0        0.39782
```
✅ All three lenses identical at layer L, KL = 0.0

In [ ]:
# Verify Run 1 files exist and show sanity check
p_agg_pud_en = lens_result_path("gpt2", "pud_en")
p_tok_pud_en = lens_result_path("gpt2", "pud_en", per_token=True)
assert p_agg_pud_en.exists(), f"Missing: {p_agg_pud_en}"
assert p_tok_pud_en.exists(), f"Missing: {p_tok_pud_en}"

df_agg_pud_en = pd.read_parquet(p_agg_pud_en)
df_tok_pud_en = pd.read_parquet(p_tok_pud_en)
print(f"Aggregate: {p_agg_pud_en.relative_to(ROOT_DIR)} | {len(df_agg_pud_en)} rows")
print(f"Per-token: {p_tok_pud_en.relative_to(ROOT_DIR)} | {len(df_tok_pud_en)} rows, {df_tok_pud_en['seq_id'].nunique()} sequences")

print("\nLayer L=12 Sanity Check:")
print(df_agg_pud_en[df_agg_pud_en["layer"] == 12][["lens", "top1_acc", "kl_mean", "upos_top1_acc"]].to_string(index=False))

## 2. Run 2: `gpt2` on `wikitext` (Logit, Tuned, Jacobian Lenses)

Evaluates all three lenses on WikiText-103 test split (200 windows × 256 tokens = ~51,200 positions).
No UPOS lexicon — `upos_top1_acc` will be null per CONTRACTS.md Section 9.3.

In [2]:
# Load GPT-2 and lens components for wikitext run
spec_gpt2 = get_model_spec("gpt2")
model_gpt2, tokenizer_gpt2, meta_gpt2 = load_model(
    checkpoint=spec_gpt2["hf_id"],
    kind=spec_gpt2["kind"],
    tokenizer_kwargs={"add_prefix_space": True},
)
print(f"Loaded GPT-2 on {meta_gpt2['device']}")

tuned_gpt2 = load_official_tuned_lens(model_gpt2)
jac_gpt2 = load_jacobian_checkpoint("gpt2")
print(f"Loaded Tuned Lens + Jacobian {tuple(jac_gpt2.shape)}")

W1118 16:31:41.516000 39171 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
[transformers] The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9750.98it/s]


Loaded GPT-2 on mps
Loaded Tuned Lens + Jacobian (13, 768, 768)


In [3]:
# Prepare WikiText-103 test batches (up to 200 windows of 256 tokens)
wiki_batches = prepare_wikitext_batches(
    tokenizer_gpt2,
    window_size=256,
    max_windows=200,
    batch_size=4,  # reduced from 8 to fit MPS unified memory
)
print(f"Prepared {len(wiki_batches)} WikiText-103 batches")

# Flush MPS cache before the memory-intensive evaluation
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Run evaluation (no upos_lexicon — wikitext has no UD annotation)
print("Running full evaluation on (gpt2, wikitext)...")
df_agg_wiki, df_tok_wiki = evaluate_lenses(
    model_gpt2,
    tokenizer_gpt2,
    model_name="gpt2",
    dataset_name="wikitext",
    data_batches=wiki_batches,
    tuned_translators=tuned_gpt2,
    jacobian_matrices=jac_gpt2,
    upos_lexicon=None,
    lenses=("logit", "tuned", "jacobian"),
    max_token_sequences=100,
)

# Save Parquet files
agg_path_wiki = lens_result_path("gpt2", "wikitext")
tok_path_wiki = lens_result_path("gpt2", "wikitext", per_token=True)
df_agg_wiki.to_parquet(agg_path_wiki, index=False)
df_tok_wiki.to_parquet(tok_path_wiki, index=False)
print(f"✅ Aggregate → {agg_path_wiki.relative_to(ROOT_DIR)} ({len(df_agg_wiki)} rows)")
print(f"✅ Per-token → {tok_path_wiki.relative_to(ROOT_DIR)} ({len(df_tok_wiki)} rows)")

# Sanity check at layer L
layer_L_wiki = df_agg_wiki[df_agg_wiki["layer"] == 12][["lens", "top1_acc", "kl_mean", "upos_top1_acc"]]
print("\nLayer L=12 Sanity Check (upos_top1_acc must be null):")
print(layer_L_wiki.to_string(index=False))

# Free GPT-2 from memory before loading GePpeTto
del model_gpt2, tuned_gpt2, jac_gpt2, wiki_batches
gc.collect()
print("\n🧹 GPT-2 freed from memory.")

Prepared 50 WikiText-103 batches
Running full evaluation on (gpt2, wikitext)...
  Processed batch  10/50 in 598.2s
  Processed batch  20/50 in 1183.8s
  Processed batch  30/50 in 1507.1s
  Processed batch  40/50 in 1567.8s
  Processed batch  50/50 in 1627.8s
✅ Aggregate → results/lens/gpt2__wikitext.parquet (39 rows)
✅ Per-token → results/lens/tokens/gpt2__wikitext.parquet (994500 rows)

Layer L=12 Sanity Check (upos_top1_acc must be null):
    lens  top1_acc  kl_mean upos_top1_acc
   logit  0.329098      0.0          None
   tuned  0.329098      0.0          None
jacobian  0.329098      0.0          None

🧹 GPT-2 freed from memory.


## 3. Run 3: `geppetto` on `pud_it` (Logit, Tuned, Jacobian Lenses)

Evaluates all three lenses on PUD IT using:
- Trained Tuned Lens (`checkpoints/tuned_lens/geppetto.pt`)
- Estimated Jacobians (`checkpoints/jacobian_lens/geppetto.pt`)
- ISDT train UPOS bridge lexicon (per CONTRACTS.md §10: use ISDT for GePpeTto)

In [3]:
# Load GePpeTto
spec_it = get_model_spec("geppetto")
model_it, tokenizer_it, meta_it = load_model(
    checkpoint=spec_it["hf_id"],
    kind=spec_it["kind"],
    tokenizer_kwargs={"add_prefix_space": True},
)
print(f"Loaded GePpeTto on {meta_it['device']}")

# Tuned Lens checkpoint + Jacobian checkpoint
# load_tuned_lens_checkpoint returns (TunedLensTranslators, metadata_dict)
tuned_it, _ = load_tuned_lens_checkpoint("geppetto")
tuned_it = tuned_it.to(meta_it["device"])
jac_it = load_jacobian_checkpoint("geppetto")
print(f"Loaded GePpeTto Tuned Lens + Jacobian {tuple(jac_it.shape)}")

# ISDT train UPOS lexicon
isdt_train = load_word_table("isdt", "train")
isdt_lexicon = build_upos_lexicon(isdt_train)
print(f"ISDT Lexicon: {len(isdt_lexicon):,} entries")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7490.64it/s]


Loaded GePpeTto on mps
Loaded GePpeTto Tuned Lens + Jacobian (13, 768, 768)
ISDT Lexicon: 33,216 entries


In [4]:
# Prepare PUD IT batches
pud_it_batches = prepare_pud_batches("pud_it", tokenizer_it, batch_size=16)
print(f"Prepared {len(pud_it_batches)} PUD IT batches")

# Run evaluation
print("Running full evaluation on (geppetto, pud_it)...")
df_agg_pud_it, df_tok_pud_it = evaluate_lenses(
    model_it,
    tokenizer_it,
    model_name="geppetto",
    dataset_name="pud_it",
    data_batches=pud_it_batches,
    tuned_translators=tuned_it,
    jacobian_matrices=jac_it,
    upos_lexicon=isdt_lexicon,
    lenses=("logit", "tuned", "jacobian"),
    max_token_sequences=100,
)

# Save Parquet files
agg_path_pud_it = lens_result_path("geppetto", "pud_it")
tok_path_pud_it = lens_result_path("geppetto", "pud_it", per_token=True)
df_agg_pud_it.to_parquet(agg_path_pud_it, index=False)
df_tok_pud_it.to_parquet(tok_path_pud_it, index=False)
print(f"✅ Aggregate → {agg_path_pud_it.relative_to(ROOT_DIR)} ({len(df_agg_pud_it)} rows)")
print(f"✅ Per-token → {tok_path_pud_it.relative_to(ROOT_DIR)} ({len(df_tok_pud_it)} rows)")

# Sanity check at layer L=12
layer_L_it = df_agg_pud_it[df_agg_pud_it["layer"] == 12][["lens", "top1_acc", "kl_mean", "upos_top1_acc"]]
print("\nLayer L=12 Sanity Check (all lenses must equal, KL=0.0):")
print(layer_L_it.to_string(index=False))

Prepared 63 PUD IT batches
Running full evaluation on (geppetto, pud_it)...
  Processed batch  10/63 in 235.9s
  Processed batch  20/63 in 297.6s
  Processed batch  30/63 in 373.1s
  Processed batch  40/63 in 453.5s
  Processed batch  50/63 in 527.2s
  Processed batch  60/63 in 593.9s
  Processed batch  63/63 in 615.3s
✅ Aggregate → results/lens/geppetto__pud_it.parquet (39 rows)
✅ Per-token → results/lens/tokens/geppetto__pud_it.parquet (120666 rows)

Layer L=12 Sanity Check (all lenses must equal, KL=0.0):
    lens  top1_acc  kl_mean  upos_top1_acc
   logit  0.226824      0.0       0.357635
   tuned  0.226824      0.0       0.357635
jacobian  0.226824      0.0       0.357635


## 4. Final Verification — All 6 Files + Acceptance Criteria

Verifies that all 6 Parquet files exist with the exact contract columns and pass the layer L sanity check.

In [1]:
EXPECTED_AGG_COLS = ["model", "dataset", "lens", "layer", "L", "tau",
                     "top1_acc", "entropy_mean", "kl_mean", "upos_top1_acc", "n_positions"]
EXPECTED_TOK_COLS = ["seq_id", "pos", "token", "target_token",
                     "lens", "layer", "top1_token", "top1_prob", "entropy", "kl"]

runs = [
    ("gpt2",     "pud_en",  12),
    ("gpt2",     "wikitext", 12),
    ("geppetto", "pud_it",  12),
]

print("Summary of Generated Parquet Result Files:")
print("-" * 90)
all_ok = True
for model_key, dataset_key, L in runs:
    p_agg = lens_result_path(model_key, dataset_key)
    p_tok = lens_result_path(model_key, dataset_key, per_token=True)

    if not p_agg.exists() or not p_tok.exists():
        print(f"❌ ({model_key}, {dataset_key}): files missing!")
        all_ok = False
        continue

    df_a = pd.read_parquet(p_agg)
    df_t = pd.read_parquet(p_tok)

    # Column check
    assert list(df_a.columns) == EXPECTED_AGG_COLS, f"Wrong agg columns for ({model_key}, {dataset_key}): {list(df_a.columns)}"
    assert list(df_t.columns) == EXPECTED_TOK_COLS, f"Wrong tok columns for ({model_key}, {dataset_key}): {list(df_t.columns)}"

    # Layer L sanity check
    layer_L_rows = df_a[df_a["layer"] == L]
    acc_vals = layer_L_rows["top1_acc"].tolist()
    kl_vals  = layer_L_rows["kl_mean"].tolist()
    acc_ok = len(set(round(v, 8) for v in acc_vals)) == 1
    kl_ok  = all(abs(v) < 1e-6 for v in kl_vals)

    # UPOS check
    if dataset_key == "wikitext":
        upos_ok = df_a["upos_top1_acc"].isna().all()
        upos_str = "upos=null ✅" if upos_ok else "upos=NOT null ❌"
    else:
        upos_ok = df_a["upos_top1_acc"].notna().any()
        upos_str = f"upos_at_L={layer_L_rows['upos_top1_acc'].values[0]:.4f} ✅" if upos_ok else "upos=null ❌"

    status = "✅" if (acc_ok and kl_ok and upos_ok) else "❌"
    print(f"{status} ({model_key}, {dataset_key}):")
    print(f"   Aggregate: {p_agg.relative_to(ROOT_DIR)} | {len(df_a)} rows")
    print(f"   Per-token: {p_tok.relative_to(ROOT_DIR)} | {len(df_t)} rows, {df_t['seq_id'].nunique()} sequences")
    print(f"   Layer L={L}: top1_acc same={acc_ok} ({acc_vals[0]:.6f}), KL=0 ={kl_ok}, {upos_str}")
    if not (acc_ok and kl_ok and upos_ok):
        all_ok = False

print("-" * 90)
if all_ok:
    print("✅ All 6 files generated and all acceptance criteria passed!")
else:
    print("❌ Some criteria failed — check output above.")

Summary of Generated Parquet Result Files:
------------------------------------------------------------------------------------------


NameError: name 'lens_result_path' is not defined